In [ ]:
from langgraph.graph import START, END, StateGraph, add_messages, MessagesState
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage
from typing import Literal

define the nodes


In [ ]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="qwen2.5-3b-instruct",
    temperature=0,
    max_completion_tokens=250
)

In [ ]:
# ask_question node
def ask_question(state: MessagesState) -> MessagesState:

    print(f"\n------->  ENTERING ask_question:")
    for i in state["messages"]:
        i.pretty_print()

    question = "what is your question?"
    print(question)

    return MessagesState(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
#chatbot node
def chatbot(state: MessagesState) -> MessagesState:  #this function is good when the number of nodes increasing
    
    print(f"\n------->  ENTERING chatbot:")
    for i in state["messages"]:
        i.pretty_print()

    response = llm.invoke(state["messages"])
    response.pretty_print()

    return MessagesState(messages= [response])

In [ ]:
# ask_another_question node 
def ask_another_question(state: MessagesState) -> MessagesState:

    print(f"\n------->  ENTERING ask_another_question:")
    for i in state["messages"]:
        i.pretty_print()

    question = "would you like to ask another question (yes/no)?"
    print(question)

    return MessagesState(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
# Trimming node
def trim_messages(state: MessagesState) -> MessagesState:

    print(f"\n------->  ENTERING trim_messages:")

    remove_messages = [RemoveMessage(id= i.id) for i in state["messages"][:-5]]

    return MessagesState(messages= remove_messages)

In [ ]:
# Define routing function, conditional edges
def routing_function(state: MessagesState) -> Literal["trim_messages", "__end__"]: #or can put str instead of Literal[...]

    if state["messages"][-1].content == "yes": #check last message to append new message, change from 0 to -1
        return "trim_messages"
    else:
        return "__end__"

In [ ]:
graph = StateGraph(MessagesState)

In [ ]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("trim_messages", trim_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_edge("trim_messages", "ask_question")
graph.add_conditional_edges(source= "ask_another_question", 
                            path= routing_function)

In [ ]:
graph_compiled = graph.compile() 
graph_compiled.invoke(MessagesState(messages=[]))